# AgentMorph - Stage 3 Full Sweep - **Phi-4**

Per-model notebook. Runs **one model** (this one: `Phi-4`) x 2 frameworks x 10 rules x 20 scenarios on a fresh Colab session.

**Why per-model:** each notebook is a fresh Python process, so VRAM from the previous model can't leak into this one (we hit an actual OOM doing sequential model loads in a single process during the dress rehearsal). All 5 notebooks share `/content/drive/MyDrive/AgentMorph/runs/stage3_baseline/` on Drive; the runner's manifest key includes `model_id` so there are no file collisions.

**Recommended launch order (smallest -> largest VRAM on T4):**
  1. `stage3_full_sweep_Llama-3.2-3B.ipynb`
  2. `stage3_full_sweep_Qwen2.5-7B.ipynb`
  3. `stage3_full_sweep_Llama-3.1-8B.ipynb`
  4. `stage3_full_sweep_Gemma-2-9B.ipynb`
  5. `stage3_full_sweep_Phi-4.ipynb` **<- you are here**

**Per-model wall-clock on T4 (4-bit):** 2-5 h.

**Scope (THIS notebook only):**
- 1 model: `Phi-4`
- 2 frameworks: native + langgraph (smolagents excluded; blows T4 context)
- 10 rules: all of `RULE_IDS`
- 1 env: ecommerce, all 20 scenarios
- **Expected cells: ~324** (2 fw x (9 rules x 18 non-refusal + 1 rule x 2 refusal))

**Before you run:** Runtime > Change runtime type > **T4 GPU**. Have your HF token ready for Section 3. The dress rehearsal (`stage3_dress_rehearsal.ipynb`) must be green before launching any full-sweep notebook.

## 1. GPU sanity check

In [ ]:
import torch
if not torch.cuda.is_available():
    print('WARN: no CUDA. Full sweep on CPU is infeasible - switch to T4 and restart.')
else:
    dev = torch.cuda.get_device_properties(0)
    print(f'CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)')

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

Paste your token, run, then **clear the token** before sharing this notebook.

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Per-model sweeps only need native + langgraph + gemini (no smolagents).
!pip install -q -e /content/AgentMorph[models,langgraph,gemini]

## 5. Verify the paraphrase cache

Hard dependency: rules 2, 3, 5 all raise `ParaphraseCacheMiss` in offline mode without this. Expect 30 + 20 + 4 = 54 entries across 3 files.

In [ ]:
import pathlib
cache_dir = pathlib.Path('/content/AgentMorph/runs/paraphrase_cache')
if not cache_dir.exists():
    raise SystemExit('paraphrase cache missing - re-run stage2_seed_paraphrases.ipynb')
files = sorted(cache_dir.glob('*.jsonl'))
assert len(files) == 3, f'expected 3 cache files, got {len(files)}'
counts = {p.name: sum(1 for _ in p.open()) for p in files}
for k, v in counts.items():
    print(f'{k}: {v} entries')
assert counts.get('schema_paraphrase_invariance.jsonl', 0) >= 30, 'schema cache incomplete'
assert counts.get('synonym_robustness.jsonl', 0) >= 20, 'synonym cache incomplete'
assert counts.get('refusal_consistency.jsonl', 0) >= 4, 'refusal cache incomplete'
print('cache OK')

## 6. Sweep parameters (this notebook runs `Phi-4` only)

In [ ]:
OUT_DIR = "/content/drive/MyDrive/AgentMorph/runs/stage3_baseline"
HF_CACHE = "/content/drive/MyDrive/AgentMorph/hf_cache"
MODEL = "Phi-4"
FRAMEWORKS = ["native", "langgraph"]
N_SCENARIOS = 20
print('per-model sweep config:')
print('  model     :', MODEL)
print('  frameworks:', FRAMEWORKS)
print('  rules     : all 10 (RULE_IDS default)')
print('  out_dir   :', OUT_DIR)

## 7. (Optional) Wipe prior cells for this model only

Mirrors the Stage-1 per-model notebook pattern. Prunes ONLY this model's rows from the shared `manifest.json` and deletes its trajectory JSONL files. Leaves other models' progress on Drive untouched. Run this ONLY after an adapter fix when you want to retry this model's cells from scratch. Skip otherwise - the runner is already idempotent on resume.

In [ ]:
import json, pathlib
RUN_DIR = pathlib.Path(OUT_DIR)
manifest_path = RUN_DIR / 'manifest.json'
traj_dir = RUN_DIR / 'trajectories'

if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    before = len(data.get('completed', {}))
    data['completed'] = {
        k: v for k, v in data.get('completed', {}).items()
        if not k.startswith(MODEL + '|')
    }
    after = len(data['completed'])
    manifest_path.write_text(json.dumps(data, indent=2))
    print(f'manifest: {before} -> {after} entries (dropped {before - after} for {MODEL})')
else:
    print('no manifest - nothing to prune')

if traj_dir.exists():
    for p in traj_dir.glob(f'{MODEL}__*.jsonl'):
        p.unlink()
        print('deleted:', p.name)
else:
    print('no trajectory dir yet')

## 8. Dry-run: verify the cell count for this model

Target: ~324 cells (2 frameworks x (9 rules x 18 non-refusal + 1 rule x 2 refusal) scenarios). Cells already done on a previous run count as `would_skip`.

In [ ]:
def _flags():
    f = ['--model', MODEL]
    for fw in FRAMEWORKS:
        f += ['--framework', fw]
    return f
flags = ' '.join(_flags())
!cd /content/AgentMorph && python -m agentmorph.runner --stage3 --dry-run \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --out-dir {OUT_DIR}

## 9. Run `Phi-4` sweep

Resumable: if Colab kills the runtime mid-sweep, re-run this cell from a fresh runtime (re-running Sections 1-4 first) and the manifest skips already-completed cells. One cell loss per kill, max.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
flags = ' '.join(_flags())
!cd /content/AgentMorph && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python -m agentmorph.runner --stage3 \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --capture-state \
    --hf-cache-dir {HF_CACHE} \
    --out-dir {OUT_DIR}

## 10. Post-run tally for `Phi-4` only

Filters the shared manifest + bugs.jsonl to just this model's rows. Run this after Section 9 finishes to see per-model progress.

In [ ]:
import json, collections, pathlib
out = pathlib.Path(OUT_DIR)
manifest = json.loads((out / 'manifest.json').read_text()) if (out / 'manifest.json').exists() else {'completed': {}}
completed = manifest.get('completed', {})

my_keys = [k for k in completed if k.startswith(MODEL + '|')]
print(f'cells for {MODEL}: {len(my_keys)}  (shared manifest total: {len(completed)})')

my_bug_keys = sum(1 for k in my_keys if completed[k].get('is_bug'))
print(f'bug cells for {MODEL} (from manifest): {my_bug_keys}')

bugs_path = out / 'bugs.jsonl'
if bugs_path.exists():
    bugs = [json.loads(l) for l in bugs_path.open() if json.loads(l).get('model_id') == MODEL]
    print(f'{MODEL} rows in bugs.jsonl: {len(bugs)}')
    if bugs:
        by_rule = collections.Counter(b['rule_id'] for b in bugs)
        by_dv = collections.Counter(b['divergence_type'] for b in bugs)
        print('\nby rule:')
        for k, v in sorted(by_rule.items()):
            print(f'  {k:38s} {v}')
        print('\nby divergence type:')
        for k, v in sorted(by_dv.items()):
            print(f'  {k:25s} {v}')

traj_dir = out / 'trajectories'
if traj_dir.exists():
    files = sorted(traj_dir.glob(f'{MODEL}__*.jsonl'))
    n_pairs = sum(sum(1 for _ in p.open()) for p in files)
    print(f'\n{MODEL} trajectory JSONL files: {len(files)}')
    print(f'{MODEL} pair records: {n_pairs}')

## 11. Peek one pair JSONL line for `Phi-4`

Quick visual confirmation that the pair record is well-formed.

In [ ]:
import json, pathlib
traj_dir = pathlib.Path(OUT_DIR) / 'trajectories'
for p in sorted(traj_dir.glob(f'{MODEL}__*.jsonl'))[:1]:
    print(f'\n=== {p.name} ===')
    line = p.open().readline()
    if not line:
        print('(empty)')
        continue
    row = json.loads(line)
    print('pair_id          :', row.get('pair_id'))
    print('rule_id          :', row.get('rule_id'))
    print('scenario_id      :', row.get('scenario_id'))
    print('is_bug           :', row.get('is_bug'))
    print('divergence_type  :', row.get('divergence_type'))
    if 'original' in row:
        o = row['original']
        print(f'original: steps={len(o["steps"])} final={str(o["final_answer"])[:80]!r}')
        m = row['mutated']
        print(f'mutated : steps={len(m["steps"])} final={str(m["final_answer"])[:80]!r}')
    elif 'trajectories' in row:
        print(f'3-way trajectories: {len(row["trajectories"])}')
        for i, t in enumerate(row['trajectories']):
            print(f'  [{i}] steps={len(t["steps"])} final={str(t["final_answer"])[:80]!r}')

## 12. Next step

**This is the last model in the recommended order.** All 5 per-model notebooks are now complete. The shared `runs/stage3_baseline/` on Drive should have ~1,640 completed cells and ~150 bug rows.

Next: bug classification (chat-side, 50-bug stratified sample), transfer study, HF dataset upload, and paper figures. Those are tracked in the 14-day plan.